# Flood event mapping using Sentinel-1 GRD data

Map the extent of flooded areas using Sentinel-1 data for pre and post event.

Generate a pre-event mosaic and use the first available image after the event. Calculate the ratio between pre and post event status and apply a threshold to get the area affected by flooding. 

In [24]:
import openeo
import rasterio
from openeo.processes import ProcessBuilder
import rioxarray as rxr
import leafmap
from IPython.display import JSON
from shapely.geometry import shape
from folium.plugins import Draw
from scipy.ndimage import uniform_filter

In [23]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [30]:
# define properties for the outputs
from pathlib import Path

out_dir = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/flood/tuscany")
event = "Tuscany_2025_S1"

## 1) Define Area of Interest and dates of the event

In [10]:
# open a map and zoom to the area of interest
m = leafmap.Map(center=(46.65, 11.4), zoom=8.5)
m

Map(center=[46.65, 11.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [11]:
feat = m.draw_features
geom_dict = feat[0]['geometry']
geom = shape(geom_dict)

minx, miny, maxx, maxy = geom.bounds

bbox = {
    "west": minx,
    "south": miny,
    "east": maxx,
    "north": maxy,
}

print(bbox)

{'west': 10.658112, 'south': 43.696673, 'east': 10.947189, 'north': 43.877108}


In [12]:
PRE_DATE  = ("2025-01-01", "2025-03-10")   # before flood
POST_DATE = ("2025-03-15", "2025-03-20")   # during/after flood

## 2) Search stac catalog for items in the given time frame and area of interest

In [13]:
from pystac_client import Client
from datetime import datetime

catalog = Client.open("https://stac.dataspace.copernicus.eu/v1")

#for c in catalog.get_collections():
#    print(c.id)

In [14]:
# Search for Sentinel-1 GRD items
search = catalog.search(
    collections=["sentinel-1-grd"],   
    bbox=[bbox["west"], bbox["south"], bbox["east"], bbox["north"]],
    datetime=f"{PRE_DATE[0]}/{POST_DATE[1]}",
    
)
items = sorted(search.items(), key=lambda i: i.datetime)

In [15]:
print(items)

[<Item id=S1A_IW_GRDH_1SDV_20250104T052752_20250104T052817_057290_070C6E_1B14_COG>, <Item id=S1A_IW_GRDH_1SDV_20250105T171448_20250105T171513_057312_070D48_FD6E_COG>, <Item id=S1A_IW_GRDH_1SDV_20250111T051943_20250111T052008_057392_071074_9604_COG>, <Item id=S1A_IW_GRDH_1SDV_20250112T170627_20250112T170652_057414_071158_BC7E_COG>, <Item id=S1A_IW_GRDH_1SDV_20250116T052752_20250116T052817_057465_071357_25AA_COG>, <Item id=S1A_IW_GRDH_1SDV_20250117T171447_20250117T171512_057487_071437_3B5F_COG>, <Item id=S1A_IW_GRDH_1SDV_20250123T051942_20250123T052007_057567_071767_26BB_COG>, <Item id=S1A_IW_GRDH_1SDV_20250124T170626_20250124T170651_057589_07184D_FAAB_COG>, <Item id=S1A_IW_GRDH_1SDV_20250128T052751_20250128T052816_057640_071A53_922F_COG>, <Item id=S1A_IW_GRDH_1SDV_20250129T171447_20250129T171512_057662_071B2C_375E_COG>, <Item id=S1A_IW_GRDH_1SDV_20250204T051942_20250204T052007_057742_071E51_3405_COG>, <Item id=S1A_IW_GRDH_1SDV_20250205T170626_20250205T170651_057764_071F35_3630_COG>, <It

## 3) Search for S1 GRD dates for pre and post event with matching orbit

In [16]:

from datetime import datetime, timezone
import numpy as np

event = datetime(2025, 3, 14, tzinfo=timezone.utc)

items = sorted(items, key=lambda i: i.datetime)

# Split before and after event
pre_candidates = [
    i for i in items 
    if i.datetime < event
]

post_candidates = [
    i for i in items 
    if i.datetime > event
]

# Find a matching orbit with enough pre-event images
orbit_pair = None

for post in sorted(post_candidates, key=lambda i: i.datetime):
    #orbit_state = post.properties["sat:orbit_state"]
    relative_orbit = post.properties["sat:relative_orbit"]

    matching_pre = [
        pre for pre in pre_candidates
        if (
            #pre.properties["sat:orbit_state"] == orbit_state
            pre.properties["sat:relative_orbit"] == relative_orbit
        )
    ]

    if len(matching_pre) >= 2:  # require multiple images
        orbit_pair = (matching_pre, post)
        break

if orbit_pair is None:
    raise RuntimeError("No matching Sentinel-1 orbit with enough pre-event images found.")

pre_images, post = orbit_pair

# Sort pre-images chronologically
pre_images = sorted(pre_images, key=lambda i: i.datetime)

print("Selected pre-event images:")
for p in pre_images:
    print(p.datetime, p.properties["sat:orbit_state"], p.properties["sat:relative_orbit"])

print("\nSelected post-event image:")
print(post.datetime, post.properties["sat:orbit_state"], post.properties["sat:relative_orbit"])

Selected pre-event images:
2025-01-04 05:27:52.913896+00:00 descending 168
2025-01-16 05:27:52.031701+00:00 descending 168
2025-01-28 05:27:51.528541+00:00 descending 168
2025-02-09 05:27:51.119810+00:00 descending 168
2025-02-21 05:27:50.505862+00:00 descending 168
2025-03-05 05:27:50.496201+00:00 descending 168

Selected post-event image:
2025-03-17 05:27:50.792128+00:00 descending 168


## 4) Define final date window for pre event and post event 

In [17]:
pre_date = [pre_images[0].datetime.date(), pre_images[-1].datetime.date()]
pre_date = (pre_date[0].strftime("%Y-%m-%d"), pre_date[1].strftime("%Y-%m-%d"))
print(f"Pre-event date range: {pre_date} ")

post_date = (
    post.datetime.strftime("%Y-%m-%d"),
    post.datetime.strftime("%Y-%m-%d")
)

print(f"Post-event date: {post_date} ")

Pre-event date range: ('2025-01-04', '2025-03-05') 
Post-event date: ('2025-03-17', '2025-03-17') 


In [ ]:
#info = connection.describe_collection("SENTINEL1_GRD")
#print(info)

## 5) Load both data cubes

In [18]:
def load_s1(temporal_extent):
    cube = connection.load_collection(
        "SENTINEL1_GRD",
        spatial_extent=bbox,
        temporal_extent=(temporal_extent),
        #temporal_extent=[temporal_extent,temporal_extent],
        bands = ["VV"],
    )
    return cube

pre_cube = load_s1(pre_date)
post_cube = load_s1(post_date)

In [19]:
pre_median = pre_cube.median_time()

In [32]:
file_pre_cube = out_dir/ f"pre_{event}.tif"
file_post_cube = out_dir/ f"post_{event}.tif"

print(file_pre_cube)
print(file_post_cube)

pre_median.download(file_pre_cube, format='GTiff')
post_cube.download(file_post_cube, format='GTiff')

/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/flood/tuscany/pre_Tuscany_2025_S1.tif
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/flood/tuscany/post_Tuscany_2025_S1.tif


## 6) Apply speckle filter to both images

### Apply LEE Filter

In [ ]:
# apply speckle filtering to the pre & the post image before calculating the ratio
def lee_filter(img, size=7):

    img = img.astype(np.float32)

    # Local mean
    mean = uniform_filter(img, size=size)

    # Local mean of squared values
    mean_sq = uniform_filter(img ** 2, size=size)

    # Local variance
    variance = mean_sq - mean ** 2

    # Estimate noise variance
    noise_variance = np.nanmean(variance)

    # Weight
    weights = np.maximum(
        variance - noise_variance,
        0
    ) / (variance + 1e-10)

    # Lee filtered image
    result = mean + weights * (img - mean)

    return result


In [62]:
with rasterio.open(file_pre_cube) as src:
    img_pre = src.read()
    profile = src.profile

filtered_pre = lee_filter(img_pre, size=7)

out_file = file_pre_cube.with_name(
    file_pre_cube.stem + "_filtered" + file_pre_cube.suffix
)

with rasterio.open(out_file, "w", **profile) as dst:
    dst.write(filtered_pre)

In [63]:
with rasterio.open(file_post_cube) as src:
    img_post = src.read()
    profile = src.profile

filtered_post = lee_filter(img_post, size=7)

out_file = file_post_cube.with_name(
    file_post_cube.stem + "_filtered" + file_post_cube.suffix
)

with rasterio.open(out_file, "w", **profile) as dst:
    dst.write(filtered_post)

In [38]:
ratio = filtered_post/img_pre

In [39]:
ratio_filename = out_dir/ f"ratio_{event}.tif"

with rasterio.open(ratio_filename, "w", **profile) as dst:
    dst.write(ratio)

### 7) Apply threshold to the result ti get only the flooded areas

In [50]:
thresh = 0.3

ratio_thresh = (ratio <= thresh).astype(np.uint8)

file_change = out_dir / f"change_{event}.tif"

with rasterio.open(file_change, "w", **profile) as dst:
    dst.write(ratio_thresh)

## Apply frost speckle filter

In [56]:
def frost_filter(img, size=7, damping=1.0):

    img = img.astype(np.float32)

    # Local mean
    mean = uniform_filter(img, size=size, mode="nearest")

    # Local mean of squared values
    mean_sq = uniform_filter(img ** 2, size=size, mode="nearest")

    # Local variance
    variance = mean_sq - mean ** 2

    # Local coefficient of variation
    cv = np.sqrt(variance) / (mean + 1e-10)

    # Create distance matrix
    radius = size // 2

    yy, xx = np.mgrid[
        -radius:radius + 1,
        -radius:radius + 1
    ]

    distance = np.sqrt(
        xx ** 2 + yy ** 2
    )

    # Adaptive Frost parameter
    alpha = damping * cv

    # For every pixel calculate weighted window
    padded = np.pad(img, radius, mode="edge")

    output = np.empty_like(img)

    for y in range(img.shape[0]):
        for x in range(img.shape[1]):

            window = padded[
                y:y + size,
                x:x + size
            ]

            weights = np.exp(
                -alpha[y, x] * distance
            )

            output[y, x] = (
                np.sum(weights * window)
                / np.sum(weights)
            )

    return output

In [ ]:
with rasterio.open(file_pre_cube) as src:
    img_pre = src.read(1)
    profile = src.profile

filtered_pre = frost_filter(img_pre, size=3, damping=1.0)

out_file = file_pre_cube.with_name(
    file_pre_cube.stem + "_filtered_frost" + file_pre_cube.suffix
)

with rasterio.open(out_file, "w", **profile) as dst:
    dst.write(filtered_pre, 1)

In [66]:
with rasterio.open(file_post_cube) as src:
    img_post = src.read(1)
    profile = src.profile

filtered_post = frost_filter(img_post, size=7, damping=1.0)

out_file = file_post_cube.with_name(
    file_post_cube.stem + "_filtered_frost" + file_post_cube.suffix
)

with rasterio.open(out_file, "w", **profile) as dst:
    dst.write(filtered_post, 1)